In [2]:
import os
import torch
import torch.nn as nn
import pickle
import numpy as np

# ==========================================================================================
# 1. MODEL MİMARİLERİNİN DOĞRU BOYUTLARLA EŞİTLENMESİ
# ==========================================================================================
class IMUAutoencoder(nn.Module):
    def __init__(self):
        super(IMUAutoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(6, 12), nn.ReLU(), nn.Linear(12, 3))
        self.decoder = nn.Sequential(nn.Linear(3, 12), nn.ReLU(), nn.Linear(12, 6), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

class NavAutoencoder(nn.Module):
    def __init__(self):
        super(NavAutoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(6, 12), nn.ReLU(), nn.Linear(12, 3))
        self.decoder = nn.Sequential(nn.Linear(3, 12), nn.ReLU(), nn.Linear(12, 6), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

# Jilet gibi düzeltilen Attitude Mimarisi (4 -> 8 -> 2 -> 8 -> 4)
class AttAutoencoder(nn.Module):
    def __init__(self):
        super(AttAutoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
        self.decoder = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 4), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

class ActAutoencoder(nn.Module):
    def __init__(self):
        super(ActAutoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
        self.decoder = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

# ==========================================================================================
# 2. MODELLERİN VE SCALER'LARIN HAFIZAYA YÜKLENMESİ
# ==========================================================================================
modeller_dir = r"C:\Users\feyza\Desktop\uav_project\models"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📂 Eğitilen tüm siber güvenlik modülleri hafızaya yükleniyor...")

try:
    # Modelleri yükle
    imu_model = IMUAutoencoder().to(device); imu_model.load_state_dict(torch.load(os.path.join(modeller_dir, "imu_autoencoder_v2_tuning.pth"))); imu_model.eval()
    nav_model = NavAutoencoder().to(device); nav_model.load_state_dict(torch.load(os.path.join(modeller_dir, "navigation_autoencoder_v2_tuning.pth"))); nav_model.eval()
    att_model = AttAutoencoder().to(device); att_model.load_state_dict(torch.load(os.path.join(modeller_dir, "attitude_autoencoder_v2_tuning.pth"))); att_model.eval()
    act_model = ActAutoencoder().to(device); act_model.load_state_dict(torch.load(os.path.join(modeller_dir, "actuator_autoencoder_v2_tuning.pth"))); act_model.eval()
    
    # Scaler'ları yükle
    with open(os.path.join(modeller_dir, "imu_scaler.pkl"), "rb") as f: imu_scaler = pickle.load(f)
    with open(os.path.join(modeller_dir, "navigation_scaler.pkl"), "rb") as f: nav_scaler = pickle.load(f)
    with open(os.path.join(modeller_dir, "attitude_scaler.pkl"), "rb") as f: att_scaler = pickle.load(f)
    with open(os.path.join(modeller_dir, "actuator_scaler.pkl"), "rb") as f: act_scaler = pickle.load(f)
    
    print("🎯 TEBRİKLER: Tüm yapay zeka katmanları ve mühürlü formüller (pkl) HATASIZ senkronize edildi!")
except Exception as e:
    print(f"❌ Yükleme sırasında hata: {str(e)}")

# ==========================================================================================
# 3. CANLI ANALİZ VE TEŞHİS FONKSİYONU
# ==========================================================================================
def anlik_otonom_analiz(ham_veri_dict):
    criterion = nn.MSELoss()
    sonuclar = {}
    
    # 1. IMU Analizi
    try:
        imu_input = np.array([[ham_veri_dict['gyro_rad[0]'], ham_veri_dict['gyro_rad[1]'], ham_veri_dict['gyro_rad[2]'],
                               ham_veri_dict['accelerometer_m_s2[0]'], ham_veri_dict['accelerometer_m_s2[1]'], ham_veri_dict['accelerometer_m_s2[2]']]])
        imu_scaled = imu_scaler.transform(imu_input)
        imu_tensor = torch.tensor(imu_scaled, dtype=torch.float32).to(device)
        with torch.no_grad():
            sonuclar['IMU_Loss'] = criterion(imu_model(imu_tensor), imu_tensor).item()
    except KeyError: sonuclar['IMU_Loss'] = 0.0

    # 2. Navigation Analizi
    try:
        nav_input = np.array([[ham_veri_dict['x'], ham_veri_dict['y'], ham_veri_dict['z'],
                               ham_veri_dict['vx'], ham_veri_dict['vy'], ham_veri_dict['vz']]])
        nav_scaled = nav_scaler.transform(nav_input)
        nav_tensor = torch.tensor(nav_scaled, dtype=torch.float32).to(device)
        with torch.no_grad():
            sonuclar['Navigation_Loss'] = criterion(nav_model(nav_tensor), nav_tensor).item()
    except KeyError: sonuclar['Navigation_Loss'] = 0.0

    # 3. Attitude Analizi
    try:
        att_input = np.array([[ham_veri_dict['q[0]'], ham_veri_dict['q[1]'], ham_veri_dict['q[2]'], ham_veri_dict['q[3]']]])
        att_scaled = att_scaler.transform(att_input)
        att_tensor = torch.tensor(att_scaled, dtype=torch.float32).to(device)
        with torch.no_grad():
            sonuclar['Attitude_Loss'] = criterion(att_model(att_tensor), att_tensor).item()
    except KeyError: sonuclar['Attitude_Loss'] = 0.0

    # 4. Actuator Analizi
    try:
        act_input = np.array([[ham_veri_dict[f'output[{i}]'] for i in range(8)]])
        act_scaled = act_scaler.transform(act_input)
        act_tensor = torch.tensor(act_scaled, dtype=torch.float32).to(device)
        with torch.no_grad():
            sonuclar['Actuator_Loss'] = criterion(act_model(act_tensor), act_tensor).item()
    except KeyError: sonuclar['Actuator_Loss'] = 0.0

    return sonuclar

print("\n🚀 Teşhis motoru fonksiyonu (`anlik_otonom_analiz`) kılçıksız hazır kanka!")

📂 Eğitilen tüm siber güvenlik modülleri hafızaya yükleniyor...
🎯 TEBRİKLER: Tüm yapay zeka katmanları ve mühürlü formüller (pkl) HATASIZ senkronize edildi!

🚀 Teşhis motoru fonksiyonu (`anlik_otonom_analiz`) kılçıksız hazır kanka!


In [3]:
import time
import pandas as pd

# Kanka, test etmek için temiz veya arızalı fark etmez, elindeki herhangi bir uçuş CSV dosyasının yolunu buraya yaz:
# Örnek olarak sensor_combined veya herhangi bir birleşik test logu olabilir.
ornek_test_csv = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train\00_02_49\00_02_49_sensor_combined_0.csv" # Kendi yoluna göre güncelle kanka

print("⏱️ Otonom Teşhis Motoru Canlı Akış Stres Testi Başlatılıyor...\n")

if not os.path.exists(ornek_test_csv):
    # Eğer o an spesifik dosya yolunu bulamazsa, test etmek için elimizle çakma bir veri sözlüğü (mock data) üretip motoru tetikleyelim:
    print("⚠️ Belirtilen CSV bulunamadı, sistem otonom 'Mock Data' üreterek motoru test ediyor...")
    mock_veri = {
        'gyro_rad[0]': 0.01, 'gyro_rad[1]': -0.02, 'gyro_rad[2]': 0.05,
        'accelerometer_m_s2[0]': 0.1, 'accelerometer_m_s2[1]': 9.81, 'accelerometer_m_s2[2]': -0.2,
        'x': 10.5, 'y': -20.2, 'z': 50.0,
        'vx': 5.2, 'vy': 0.1, 'vz': -1.2,
        'q[0]': 1.0, 'q[1]': 0.0, 'q[2]': 0.0, 'q[3]': 0.0
    }
    for i in range(8):
        mock_veri[f'output[{i}]'] = 1500.0 # Standart PWM sinyali
        
    # Motoru test et
    anlik_skorlar = anlik_otonom_analiz(mock_veri)
    print("\n📊 Motor Çıktı Analiz Raporu (Mock Data):")
    print("-" * 50)
    for modul, loss in anlik_skorlar.items():
        print(f"  🔹 {modul}: {loss:.6f}")
        
else:
    # Eğer gerçek CSV varsa, ilk 5 satırı saniye saniye akıtarak test edelim:
    df_test = pd.read_csv(ornek_test_csv).head(5)
    
    for idx, row in df_test.iterrows():
        # Satırı sözlüğe çeviriyoruz (Arayüze veri akışı simülasyonu)
        ham_satir_dict = row.to_dict()
        
        # Eğer eksik kolonlar varsa mock verilerle besleyelim ki fonksiyon patlamasın
        for i in range(8):
            if f'output[{i}]' not in ham_satir_dict: ham_satir_dict[f'output[{i}]'] = 1500.0
        if 'x' not in ham_satir_dict:
            ham_satir_dict.update({'x': 0.0, 'y': 0.0, 'z': 0.0, 'vx': 0.0, 'vy': 0.0, 'vz': 0.0})
        if 'q[0]' not in ham_satir_dict:
            ham_satir_dict.update({'q[0]': 1.0, 'q[1]': 0.0, 'q[2]': 0.0, 'q[3]': 0.0})
            
        # Teşhis motorunu ateşle kanka
        skorlar = anlik_otonom_analiz(ham_satir_dict)
        
        print(f"⏱️ [Saniye/Satır {idx+1}] Gelen Telemetri Analiz Edildi:")
        print(f"   ↳ {skorlar}")
        print("-" * 70)
        time.sleep(1) # 1 saniye bekleterek gerçek zamanlı akışı simüle ediyoruz

⏱️ Otonom Teşhis Motoru Canlı Akış Stres Testi Başlatılıyor...

⏱️ [Saniye/Satır 1] Gelen Telemetri Analiz Edildi:
   ↳ {'IMU_Loss': 5.3096083973969144e-08, 'Navigation_Loss': 1.822629542402865e-06, 'Attitude_Loss': 0.00011406966223148629, 'Actuator_Loss': 0.0005354484892450273}
----------------------------------------------------------------------


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fit

⏱️ [Saniye/Satır 2] Gelen Telemetri Analiz Edildi:
   ↳ {'IMU_Loss': 6.663012896979126e-08, 'Navigation_Loss': 1.822629542402865e-06, 'Attitude_Loss': 0.00011406966223148629, 'Actuator_Loss': 0.0005354484892450273}
----------------------------------------------------------------------


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


⏱️ [Saniye/Satır 3] Gelen Telemetri Analiz Edildi:
   ↳ {'IMU_Loss': 6.467991653380523e-08, 'Navigation_Loss': 1.822629542402865e-06, 'Attitude_Loss': 0.00011406966223148629, 'Actuator_Loss': 0.0005354484892450273}
----------------------------------------------------------------------


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


⏱️ [Saniye/Satır 4] Gelen Telemetri Analiz Edildi:
   ↳ {'IMU_Loss': 6.823080411777482e-08, 'Navigation_Loss': 1.822629542402865e-06, 'Attitude_Loss': 0.00011406966223148629, 'Actuator_Loss': 0.0005354484892450273}
----------------------------------------------------------------------


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


⏱️ [Saniye/Satır 5] Gelen Telemetri Analiz Edildi:
   ↳ {'IMU_Loss': 8.347313240619769e-08, 'Navigation_Loss': 1.822629542402865e-06, 'Attitude_Loss': 0.00011406966223148629, 'Actuator_Loss': 0.0005354484892450273}
----------------------------------------------------------------------
